In [1]:
!pip install -U transformers

## Local Inference on GPU
Model page: https://huggingface.co/state-spaces/mamba2-130m

⚠️ If the generated code snippets do not work, please open an issue on either the [model repo](https://huggingface.co/state-spaces/mamba2-130m)
			and/or on [huggingface.js](https://github.com/huggingface/huggingface.js/blob/main/packages/tasks/src/model-libraries-snippets.ts) 🙏

In [2]:
from google.colab import userdata
token = userdata.get('HF_TOKEN')

In [3]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

model_id = "AntonV/mamba2-130m-hf"
tokenizer = AutoTokenizer.from_pretrained(model_id, token=token)
model = AutoModelForCausalLM.from_pretrained(model_id, dtype="auto", token=token)
model.eval()

[transformers] The fast path is not available because one of `(selective_state_update, causal_conv1d_fn, causal_conv1d_update)` is None. Falling back to the naive implementation. To install follow https://github.com/state-spaces/mamba/#installation and https://github.com/Dao-AILab/causal-conv1d


Loading weights:   0%|          | 0/218 [00:00<?, ?it/s]

Mamba2ForCausalLM(
  (backbone): Mamba2Model(
    (embeddings): Embedding(50288, 768)
    (layers): ModuleList(
      (0-23): 24 x Mamba2Block(
        (norm): Mamba2RMSNorm()
        (mixer): Mamba2Mixer(
          (act): SiLUActivation()
          (conv1d): Conv1d(1792, 1792, kernel_size=(4,), stride=(1,), padding=(3,), groups=1792)
          (in_proj): Linear(in_features=768, out_features=3352, bias=False)
          (norm): MambaRMSNormGated()
          (out_proj): Linear(in_features=1536, out_features=768, bias=False)
        )
      )
    )
    (norm_f): Mamba2RMSNorm()
  )
  (lm_head): Linear(in_features=768, out_features=50288, bias=False)
)

In [4]:
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")
print(torch.__version__)
print(torch.version.cuda)

Using device: cuda
2.10.0+cu128
12.8


In [5]:
input_ids = tokenizer("Hey how are you doing?", return_tensors="pt")["input_ids"].to(device)
model.to(device)
output = model(input_ids)
print(output.cache_params)

DynamicCache(layers=[LinearAttentionLayer, LinearAttentionLayer, LinearAttentionLayer, LinearAttentionLayer, LinearAttentionLayer, LinearAttentionLayer, LinearAttentionLayer, LinearAttentionLayer, LinearAttentionLayer, LinearAttentionLayer, LinearAttentionLayer, LinearAttentionLayer, LinearAttentionLayer, LinearAttentionLayer, LinearAttentionLayer, LinearAttentionLayer, LinearAttentionLayer, LinearAttentionLayer, LinearAttentionLayer, LinearAttentionLayer, LinearAttentionLayer, LinearAttentionLayer, LinearAttentionLayer, LinearAttentionLayer])


In [6]:
output

Mamba2CausalLMOutput(loss=None, logits=tensor([[[-109.4034, -123.6545, -100.5855,  ..., -123.8113, -123.6328,
          -123.6537],
         [  12.6488,   -4.3739,   17.2195,  ...,   -4.1592,   -4.4718,
            -4.0873],
         [  61.6835,   47.2499,   65.0402,  ...,   47.3465,   47.1545,
            47.2796],
         [-101.0707, -118.8035,  -95.2718,  ..., -118.9733, -118.9621,
          -118.7094],
         [-106.0794, -124.9325, -100.0780,  ..., -125.0399, -124.9703,
          -124.7930],
         [ -25.4473,  -47.7910,  -26.4842,  ...,  -47.9524,  -47.9480,
           -47.8289]]], device='cuda:0', grad_fn=<UnsafeViewBackward0>), cache_params=DynamicCache(layers=[LinearAttentionLayer, LinearAttentionLayer, LinearAttentionLayer, LinearAttentionLayer, LinearAttentionLayer, LinearAttentionLayer, LinearAttentionLayer, LinearAttentionLayer, LinearAttentionLayer, LinearAttentionLayer, LinearAttentionLayer, LinearAttentionLayer, LinearAttentionLayer, LinearAttentionLayer, LinearAtte

In [7]:
cache = output.cache_params

# Inspect the first layer
layer = cache.layers[0]
print(type(layer))
print(dir(layer))

# Print all attributes with shapes
for attr in vars(layer):
    val = getattr(layer, attr)
    if hasattr(val, 'shape'):
        print(f"{attr}: {val.shape}")
    elif isinstance(val, list):
        print(f"{attr}: list of {len(val)}")
    else:
        print(f"{attr}: {type(val)} = {val}")

<class 'transformers.cache_utils.LinearAttentionLayer'>
['__abstractmethods__', '__class__', '__delattr__', '__dict__', '__dir__', '__doc__', '__eq__', '__format__', '__ge__', '__getattribute__', '__getstate__', '__gt__', '__hash__', '__init__', '__init_subclass__', '__le__', '__lt__', '__module__', '__ne__', '__new__', '__reduce__', '__reduce_ex__', '__repr__', '__setattr__', '__sizeof__', '__slots__', '__str__', '__subclasshook__', '__weakref__', '_abc_impl', 'conv_kernel_size', 'conv_states', 'crop', 'device', 'dtype', 'has_previous_state', 'is_compileable', 'is_conv_states_initialized', 'is_recurrent_states_initialized', 'lazy_initialization', 'max_batch_size', 'offload', 'prefetch', 'recurrent_states', 'reorder_cache', 'reset', 'update_conv_state', 'update_recurrent_state']
conv_states: torch.Size([1, 1792, 4])
recurrent_states: torch.Size([1, 24, 64, 128])
is_conv_states_initialized: <class 'bool'> = True
is_recurrent_states_initialized: <class 'bool'> = True
has_previous_state: 

In [8]:
from datasets import load_dataset
dataset_name = "nvidia/Nemotron-RL-Instruction-Following-MultiTurnChat-v1"
split = "train"
dataset = load_dataset(dataset_name, split=split)

In [9]:
def extract_sessions(dataset):
    sessions = []

    for sample in dataset:
        dialog = sample["responses_create_params"]["input"]
        sessions.append(dialog)

    return sessions

def build_turn_snapshots(session):
    snapshots = []

    history_text = ""
    turn_id = 0

    for turn in session:
        role = turn["role"]
        content = turn["content"]

        line = f"{role.capitalize()}: {content}\n"
        history_text += line

        snapshots.append({
            "turn_id": turn_id,
            "role": role,
            "new_text": line,
            "history_text": history_text
        })

        turn_id += 1

    return snapshots

In [10]:
sessions = extract_sessions(dataset)
states = []
session = sessions[0]
snapshots = build_turn_snapshots(session)

In [11]:
model.to(device)

Mamba2ForCausalLM(
  (backbone): Mamba2Model(
    (embeddings): Embedding(50288, 768)
    (layers): ModuleList(
      (0-23): 24 x Mamba2Block(
        (norm): Mamba2RMSNorm()
        (mixer): Mamba2Mixer(
          (act): SiLUActivation()
          (conv1d): Conv1d(1792, 1792, kernel_size=(4,), stride=(1,), padding=(3,), groups=1792)
          (in_proj): Linear(in_features=768, out_features=3352, bias=False)
          (norm): MambaRMSNormGated()
          (out_proj): Linear(in_features=1536, out_features=768, bias=False)
        )
      )
    )
    (norm_f): Mamba2RMSNorm()
  )
  (lm_head): Linear(in_features=768, out_features=50288, bias=False)
)

In [12]:
for snap in snapshots:
    role = snap["role"]
    input_text = snap["new_text"]
    turn_id = snap["turn_id"]

    inputs = tokenizer(input_text, return_tensors="pt").to(device)
    with torch.no_grad():
        output = model(**inputs, use_cache=True, labels=inputs["input_ids"])

    if role == "assistant":
        loss = output.loss.item() if output.loss is not None else 0.0
        ppl = torch.exp(torch.tensor(loss)).item() if loss > 0 else 0.0
        print(f"Turn {turn_id} - Perplexity: {ppl:.4f} - Loss : {loss:.4f}")

    cache = output.cache_params

Turn 2 - Perplexity: 28.7919 - Loss : 3.3601
Turn 4 - Perplexity: 36.0655 - Loss : 3.5853
Turn 6 - Perplexity: 39.0732 - Loss : 3.6654
Turn 8 - Perplexity: 36.3090 - Loss : 3.5921
Turn 10 - Perplexity: 33.2207 - Loss : 3.5032
Turn 12 - Perplexity: 35.1733 - Loss : 3.5603


In [13]:
cache = None
for snap in snapshots:
    role = snap["role"]
    input_text = snap["new_text"]
    turn_id = snap["turn_id"]

    inputs = tokenizer(input_text, return_tensors="pt").to(device)

    if turn_id == 0:
        with torch.no_grad():
            output = model(
                **inputs,
                use_cache=True,
                labels=inputs["input_ids"]
            )
        cache = output.cache_params
        ssm_states = [layer.recurrent_states for layer in cache.layers]
        torch.save(ssm_states, "cache.pt")


    else:
        prev_cache = torch.load("cache.pt")
        for i in range(len(prev_cache)):
            cache.layers[i].recurrent_states = prev_cache[i]
        with torch.no_grad():
            output = model(
                **inputs,
                use_cache=True,
                labels=inputs["input_ids"]
            )

        cache = output.cache_params
        ssm_states = [layer.recurrent_states for layer in cache.layers]
        torch.save(ssm_states, "cache.pt")

        if role == "assistant":
            loss = output.loss.item() if output.loss is not None else 0.0
            ppl = torch.exp(torch.tensor(loss)).item() if loss > 0 else 0.0
            print(f"Turn {turn_id} - Perplexity: {ppl:.4f} - Loss: {loss:.4f}")

Turn 2 - Perplexity: 28.7919 - Loss: 3.3601
Turn 4 - Perplexity: 36.0655 - Loss: 3.5853
Turn 6 - Perplexity: 39.0732 - Loss: 3.6654
Turn 8 - Perplexity: 36.3090 - Loss: 3.5921
Turn 10 - Perplexity: 33.2207 - Loss: 3.5032
Turn 12 - Perplexity: 35.1733 - Loss: 3.5603


In [14]:
from transformers.cache_utils import DynamicCache, LinearAttentionLayer
import torch

num_layers = 24
batch_size = 1
heads      = 24
head_dim   = 64
d_state    = 128
d_inner    = 1792
d_conv     = 4


def create_cache(num_layers=24, batch_size=1, heads=24, head_dim=64, d_state=128, d_inner=1792, d_conv=4):
  cache = DynamicCache()

  for i in range(num_layers):
      layer = LinearAttentionLayer()
      layer.recurrent_states = torch.zeros(batch_size, heads, head_dim, d_state).to(device)
      layer.conv_states      = torch.zeros(batch_size, d_inner, d_conv).to(device)
      layer.has_previous_state = False
      cache.layers.append(layer)
  return cache

In [15]:

cache = None
for snap in snapshots:
    role = snap["role"]
    input_text = snap["new_text"]
    turn_id = snap["turn_id"]

    inputs = tokenizer(input_text, return_tensors="pt").to(device)

    if turn_id == 0:
        with torch.no_grad():
            output = model(
                **inputs,
                use_cache=True,
                labels=inputs["input_ids"]
            )
        cache = output.cache_params
        ssm_states = [layer.recurrent_states for layer in cache.layers]
        torch.save(ssm_states, "cache.pt")


    else:
        prev_cache = torch.load("cache.pt")
        cache = create_cache()
        for i in range(len(prev_cache)):
            cache.layers[i].recurrent_states = prev_cache[i]

        with torch.no_grad():
            output = model(
                **inputs,
                use_cache=True,
                cache_params=cache,
                labels=inputs["input_ids"]
            )

        cache = output.cache_params
        ssm_states = [layer.recurrent_states for layer in cache.layers]
        torch.save(ssm_states, "cache.pt")

        if role == "assistant":
            loss = output.loss.item() if output.loss is not None else 0.0
            ppl = torch.exp(torch.tensor(loss)).item() if loss > 0 else 0.0
            print(f"Turn {turn_id} - Perplexity: {ppl:.4f} - Loss: {loss:.4f}")

Turn 2 - Perplexity: 28.7919 - Loss: 3.3601
Turn 4 - Perplexity: 36.0655 - Loss: 3.5853
Turn 6 - Perplexity: 39.0732 - Loss: 3.6654
Turn 8 - Perplexity: 36.3090 - Loss: 3.5921
Turn 10 - Perplexity: 33.2207 - Loss: 3.5032
Turn 12 - Perplexity: 35.1733 - Loss: 3.5603


In [16]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [17]:
import model_loader, state_utils, evaluate as evaluate_module, data, autoencoder, plot, utils

In [18]:
config = utils.read_config("configs/config1.yaml")
config["data"]["max_length"] = 1024

paths = config["paths"]
output_dir = paths["output_dir"]
text_history_dir = paths["text_history_dir"]+"/history.txt"
state_dir = paths["state_dir"]+"/state.pt"
plot_dir = paths["plot_dir"]

In [19]:
max_seq_length = config["data"]["max_length"]

In [20]:
import copy
import yaml
import torch
import pandas as pd
import csv
import time
import numpy as np
NUM_RUNS = 5
experiment1_path = output_dir + "/experiment1/experiment1.csv"

In [21]:
def run_baseline(model, tokenizer, snapshots, device, text_history_dir):
    output_data = {}
    for snap in snapshots:
        turn_id = snap["turn_id"]
        history_text = ""

        if turn_id > 0:
            history_text = utils.load_text(text_history_dir)

        combined_text = utils.concatenate_texts([history_text, snap["new_text"]])

        encoded_input = tokenizer(combined_text, max_length=max_seq_length, truncation=True, return_tensors="pt")
        truncated_input_ids = encoded_input["input_ids"]
        truncated_combined_text = tokenizer.decode(truncated_input_ids[0], skip_special_tokens=True)

        _, baseline_latency, baseline_ppl = evaluate_module.evaluate_baseline(
            model,
            tokenizer,
            truncated_combined_text,
            device=device
        )
        if snap["role"] == "assistant":
            baseline_size_kb = utils.get_memory_size_kb(text_history_dir)
            output_data[turn_id] = {
                "baseline_latency": baseline_latency,
                "baseline_size_kb": baseline_size_kb,
                "baseline_ppl": baseline_ppl
            }
        utils.save_text(truncated_combined_text, text_history_dir)
    return output_data

def run_state_management(model, tokenizer, snapshots, device, state_dir):
    output_data = {}
    for snap in snapshots:
        turn_id = snap["turn_id"]

        if turn_id == 0:
            state_output, state_latency, state_ppl = evaluate_module.evaluate_baseline(
                model,
                tokenizer,
                snap["new_text"],
                device=device
            )
        else:
            previous_state = state_utils.load_state(state_dir)

            state_output, state_latency, state_ppl = evaluate_module.evaluate(
                model,
                tokenizer,
                snap["new_text"],
                previous_state,
                device=device
            )

        new_state = [layer.recurrent_states for layer in state_output.cache_params.layers]
        state_utils.save_state(new_state, state_dir)
        if snap["role"] == "assistant":
            state_size_kb = utils.get_memory_size_kb(state_dir)
            output_data[turn_id] = {
                "state_latency": state_latency,
                "state_size_kb": state_size_kb,
                "state_ppl": state_ppl
            }
    return output_data

In [22]:
print("Starting first run of Experiment 1...")
torch.cuda.empty_cache()
print("starting baseline run...")
baseline_output = run_baseline(model, tokenizer, snapshots, device, text_history_dir)
print("starting state management run...")
state_output = run_state_management(model, tokenizer, snapshots, device, state_dir)
print("Completed first run of Experiment 1.")
for _ in range(NUM_RUNS-1):
    torch.cuda.empty_cache()
    print(f"Run {_+2}/{NUM_RUNS}...")
    print("starting baseline run...")
    baseline_data = run_baseline(model, tokenizer, snapshots, device, text_history_dir)
    print("starting state management run...")
    state_data = run_state_management(model, tokenizer, snapshots, device, state_dir)
    print("aggregating results...")
    for turn_id in baseline_data:
        baseline_output[turn_id]["baseline_latency"] += baseline_data[turn_id]["baseline_latency"]
        baseline_output[turn_id]["baseline_size_kb"] += baseline_data[turn_id]["baseline_size_kb"]
        baseline_output[turn_id]["baseline_ppl"] += baseline_data[turn_id]["baseline_ppl"]

        state_output[turn_id]["state_latency"] += state_data[turn_id]["state_latency"]
        state_output[turn_id]["state_size_kb"] += state_data[turn_id]["state_size_kb"]
        state_output[turn_id]["state_ppl"] += state_data[turn_id]["state_ppl"]

    print(f"Completed run {_+2}/{NUM_RUNS}")

for turn_id in baseline_output:
    baseline_output[turn_id]["baseline_latency"] /= NUM_RUNS
    baseline_output[turn_id]["baseline_size_kb"] /= NUM_RUNS
    baseline_output[turn_id]["baseline_ppl"] /= NUM_RUNS

    state_output[turn_id]["state_latency"] /= NUM_RUNS
    state_output[turn_id]["state_size_kb"] /= NUM_RUNS
    state_output[turn_id]["state_ppl"] /= NUM_RUNS

with open(experiment1_path, "w", newline="", encoding="utf-8") as f:
    writer = csv.writer(f)
    writer.writerow(["turn", "baseline_latency", "state_latency", "txt_size_kb", "pt_size_kb", "baseline_ppl", "state_ppl"])
    for turn_id in baseline_output:
        writer.writerow([
            turn_id,
            baseline_output[turn_id]["baseline_latency"],
            state_output[turn_id]["state_latency"],
            baseline_output[turn_id]["baseline_size_kb"],
            state_output[turn_id]["state_size_kb"],
            baseline_output[turn_id]["baseline_ppl"],
            state_output[turn_id]["state_ppl"]
        ])

df = pd.read_csv(experiment1_path)
latency_df = df[["turn", "baseline_latency", "state_latency"]]
size_df = df[["turn", "txt_size_kb", "pt_size_kb"]]
plot.plot_memory_growth(size_df, plot_dir + "/experiment1/memory_growth.png")
plot.plot_latency_comparison(latency_df, plot_dir + "/experiment1/latency_comparison.png")
plot.plot_ppl_comparison(df[["turn", "baseline_ppl", "state_ppl"]], plot_dir + "/experiment1/perplexity_comparison.png")

Starting first run of Experiment 1...


[transformers] Ignoring clean_up_tokenization_spaces=True for BPE tokenizer GPTNeoXTokenizer. The clean_up_tokenization post-processing step is designed for WordPiece tokenizers and is destructive for BPE (it strips spaces before punctuation). Set clean_up_tokenization_spaces=False to suppress this warning, or set clean_up_tokenization_spaces_for_bpe_even_though_it_will_corrupt_output=True to force cleanup anyway.


starting baseline run...
starting state management run...
Completed first run of Experiment 1.
Run 2/5...
starting baseline run...
starting state management run...
aggregating results...
Completed run 2/5
Run 3/5...
starting baseline run...
starting state management run...
aggregating results...
Completed run 3/5
Run 4/5...
starting baseline run...
starting state management run...
aggregating results...
Completed run 4/5
Run 5/5...
starting baseline run...
starting state management run...
aggregating results...
Completed run 5/5


In [23]:
print(f"Loading state from: {state_dir}")
saved_recurrent_states = state_utils.load_state(state_dir)

print(f"Number of layers (recurrent states saved): {len(saved_recurrent_states)}")
for i, recurrent_state in enumerate(saved_recurrent_states):
    print(f"  Layer {i}: Recurrent state shape: {recurrent_state.shape}")

print("\nNote: The `state.pt` file only contains `recurrent_states`. The `conv_states` were the part of the model's cache that needed to be explicitly initialized when the cache was reconstructed, which was addressed in the previous fix.")

Loading state from: results/states/state.pt
Number of layers (recurrent states saved): 24
  Layer 0: Recurrent state shape: torch.Size([1, 24, 64, 128])
  Layer 1: Recurrent state shape: torch.Size([1, 24, 64, 128])
  Layer 2: Recurrent state shape: torch.Size([1, 24, 64, 128])
  Layer 3: Recurrent state shape: torch.Size([1, 24, 64, 128])
  Layer 4: Recurrent state shape: torch.Size([1, 24, 64, 128])
  Layer 5: Recurrent state shape: torch.Size([1, 24, 64, 128])
  Layer 6: Recurrent state shape: torch.Size([1, 24, 64, 128])
  Layer 7: Recurrent state shape: torch.Size([1, 24, 64, 128])
  Layer 8: Recurrent state shape: torch.Size([1, 24, 64, 128])
  Layer 9: Recurrent state shape: torch.Size([1, 24, 64, 128])
  Layer 10: Recurrent state shape: torch.Size([1, 24, 64, 128])
  Layer 11: Recurrent state shape: torch.Size([1, 24, 64, 128])
  Layer 12: Recurrent state shape: torch.Size([1, 24, 64, 128])
  Layer 13: Recurrent state shape: torch.Size([1, 24, 64, 128])
  Layer 14: Recurrent st

Experiment 2:

In [24]:
import torch
import torch.nn as nn


class Autoencoder(nn.Module):
    def __init__(self, head_dim, d_state, hidden_dim):
        super().__init__()
        self.hidden_dim = hidden_dim
        self.head_dim   = head_dim
        self.d_state    = d_state
        self.input_dim  = head_dim * d_state  # 64*128 = 8192

        self.encoder_net = nn.Sequential(
            nn.Linear(self.input_dim, 256),
            nn.ReLU(),
            nn.Linear(256, self.hidden_dim)
        )
        self.decoder_net = nn.Sequential(
            nn.Linear(self.hidden_dim, 256),
            nn.ReLU(),
            nn.Linear(256, self.input_dim)
        )

    def encoder(self, x):
        return self.encoder_net(x)

    def decoder(self, z):
        return self.decoder_net(z)

    def forward(self, x):
        z = self.encoder(x)
        reconstructed = self.decoder(z)
        return reconstructed, z

    def fit(self, states, num_epochs=10, batch_size=256, learning_rate=1e-3, device="cpu"):
        self.to(device)
        self.train()

        # Flatten to [num_samples * heads, head_dim * d_state]
        num_samples = states.shape[0]
        data = states.detach().float()
        data = data.view(-1, self.input_dim).to(device)

        criterion = nn.MSELoss()
        optimizer = torch.optim.Adam(self.parameters(), lr=learning_rate)

        loss_history = []

        for epoch in range(num_epochs):
            perm = torch.randperm(data.size(0))
            data = data[perm]

            epoch_loss  = 0.0
            num_batches = 0

            for i in range(0, data.size(0), batch_size):
                batch = data[i:i + batch_size]

                optimizer.zero_grad()
                z            = self.encoder(batch)
                reconstructed = self.decoder(z)
                loss         = criterion(reconstructed, batch)
                loss.backward()
                optimizer.step()

                epoch_loss  += loss.item()
                num_batches += 1

            avg_loss = epoch_loss / num_batches
            loss_history.append(avg_loss)
            print(f"  Epoch [{epoch+1:>3}/{num_epochs}] Loss: {avg_loss:.6f}")

        return loss_history

In [25]:
training_sessions = sessions[1]

train_snapshots = data.build_turn_snapshots(training_sessions)

In [26]:
num_layers = 24
sample_states = {i: [] for i in range(num_layers)}

In [27]:
for snap in train_snapshots:
    inputs = tokenizer(snap["new_text"], return_tensors="pt").to(device)

    with torch.no_grad():
        output = model(
            **inputs,
            use_cache=True
        )

    cache = output.cache_params

    for i, layer in enumerate(cache.layers):
        state = layer.recurrent_states          # [1, 24, 64, 128]
        sample_states[i].append(
            state.squeeze(0).cpu()              # [24, 64, 128] — remove batch dim
        )

    torch.cuda.empty_cache()

In [28]:
latent_dims = [16, 32, 64, 128]

ae_experiments = {
    ld: nn.ModuleList([
        Autoencoder(head_dim=64, d_state=128, hidden_dim=ld)
        for _ in range(num_layers)
    ])
    for ld in latent_dims
}

ae_list = ae_experiments[32]   # 24 AEs with latent_dim=32
ae_list = ae_experiments[64]   # 24 AEs with latent_dim=64

In [36]:
stacked_states = {
    i: torch.stack(sample_states[i], dim=0)   # [num_samples, heads, head_dim, d_state]
    for i in range(num_layers)
}

for latent_dim, ae_list in ae_experiments.items():
    for layer_idx in range(num_layers):
        print(f"\n{'='*40}")
        print(f"Training AE for Layer {layer_idx} , latent-dim {latent_dim}")
        print(f"{'='*40}")

        states = stacked_states[layer_idx]
        print(f"  States shape: {states.shape}")   # [num_samples, 24, 64, 128]

        loss_history = ae_list[layer_idx].fit(
            states,
            num_epochs=20,
            batch_size=256,
            learning_rate=1e-3,
            device=device
        )


Training AE for Layer 0 , latent-dim 16
  States shape: torch.Size([30, 24, 64, 128])
  Epoch [  1/20] Loss: 0.051633
  Epoch [  2/20] Loss: 0.048532
  Epoch [  3/20] Loss: 0.045360
  Epoch [  4/20] Loss: 0.042972
  Epoch [  5/20] Loss: 0.039743
  Epoch [  6/20] Loss: 0.035190
  Epoch [  7/20] Loss: 0.031005
  Epoch [  8/20] Loss: 0.027524
  Epoch [  9/20] Loss: 0.025065
  Epoch [ 10/20] Loss: 0.022408
  Epoch [ 11/20] Loss: 0.019793
  Epoch [ 12/20] Loss: 0.017591
  Epoch [ 13/20] Loss: 0.015595
  Epoch [ 14/20] Loss: 0.013983
  Epoch [ 15/20] Loss: 0.012497
  Epoch [ 16/20] Loss: 0.011251
  Epoch [ 17/20] Loss: 0.010152
  Epoch [ 18/20] Loss: 0.009121
  Epoch [ 19/20] Loss: 0.008228
  Epoch [ 20/20] Loss: 0.007432

Training AE for Layer 1 , latent-dim 16
  States shape: torch.Size([30, 24, 64, 128])
  Epoch [  1/20] Loss: 0.308822
  Epoch [  2/20] Loss: 0.291575
  Epoch [  3/20] Loss: 0.282942
  Epoch [  4/20] Loss: 0.260497
  Epoch [  5/20] Loss: 0.231339
  Epoch [  6/20] Loss: 0.1

In [30]:
def run_autoencoder(model, tokenizer, snapshots, device, state_dir, ae_list):
    output_data = {}
    for ae in ae_list:
        ae.eval()
        ae.to(device)
    for snap in snapshots:
        turn_id = snap["turn_id"]

        if turn_id == 0:
            state_output, state_latency, state_ppl = evaluate_module.evaluate_baseline(
                model,
                tokenizer,
                snap["new_text"],
                device=device
            )
        else:
            compressed_state = state_utils.load_state(state_dir)

            decompressed_state = []
            for layer_idx, latent in enumerate(compressed_state):
                latent = latent.to(device)                          # [heads, latent_dim]
                latent = latent.unsqueeze(0)                        # [1, heads, latent_dim]
                reconstructed = ae_list[layer_idx].decoder(latent)  # [1, heads, head_dim, d_state]
                decompressed_state.append(reconstructed)

            state_output, state_latency, state_ppl = evaluate_module.evaluate(
                model,
                tokenizer,
                snap["new_text"],
                decompressed_state,
                device=device
            )

        raw_states = [layer.recurrent_states for layer in state_output.cache_params.layers]
        compressed = []
        for layer_idx, state in enumerate(raw_states):
            state = state.to(device)                                # [1, heads, head_dim, d_state]
            latent = ae_list[layer_idx].encoder(
                state.view(1, 24, -1)                               # [1, heads, head_dim*d_state]
            )                                                        # [1, heads, latent_dim]
            compressed.append(latent.squeeze(0).cpu())              # [heads, latent_dim]

        state_utils.save_state(compressed, state_dir)

        if snap["role"] == "assistant":
            state_size_kb = utils.get_memory_size_kb(state_dir)
            output_data[turn_id] = {
                "state_latency": state_latency,
                "state_size_kb": state_size_kb,
                "state_ppl": state_ppl
            }

    return output_data

In [31]:
def run_experiment_2(
    model, tokenizer, snapshots, ae_experiments,  # ae_experiments: {latent_dim: ae_list}
    output_dir, experiment_2_benchmark_path, plot_dir,
    device
):
    # Write header once
    with open(experiment_2_benchmark_path, "w", newline="") as f:
        writer = csv.writer(f)
        writer.writerow([
            "turn", "state_latency", "compressed_latency",
            "state_ppl", "compressed_ppl",
            "original_size_kb", "compressed_size_kb",
            "autoencoder_latent_dim"
        ])

    print("Running original state management...")
    original_dir     = f"{output_dir}/state_original.pt"
    original_results = run_state_management(
        model, tokenizer, snapshots, device, original_dir
    )

    for latent_dim, ae_list in ae_experiments.items():
        print(f"\n{'='*50}")
        print(f"Running AE latent_dim={latent_dim}")
        print(f"{'='*50}")

        compress_dir = f"{output_dir}/state_compressed_{latent_dim}.pt"

        ae_results = run_autoencoder(
            model, tokenizer, snapshots, device, compress_dir, ae_list
        )

        with open(experiment_2_benchmark_path, "a", newline="") as f:  # ← "a" not "w"
            writer = csv.writer(f)
            for turn_id in original_results:
                if turn_id not in ae_results:
                    continue
                orig = original_results[turn_id]
                comp = ae_results[turn_id]
                writer.writerow([
                    turn_id,
                    orig["state_latency"], comp["state_latency"],
                    orig["state_ppl"],     comp["state_ppl"],
                    orig["state_size_kb"], comp["state_size_kb"],
                    latent_dim
                ])

        torch.cuda.empty_cache()

    df = pd.read_csv(experiment_2_benchmark_path)
    plot.plot_perplexity_comparison(df, plot_dir)
    plot.plot_latency_comparison_exp2(df, plot_dir)
    plot.plot_memory_growth_exp2(df, plot_dir)
    return df

In [32]:
experiment2_path = output_dir + "/experiment2/experiment2.csv"


In [33]:
df = run_experiment_2(
    model, tokenizer, snapshots, ae_experiments,
    output_dir + "/experiment2",
    experiment2_path,
    plot_dir + "/experiment2",
    device
)

Running original state management...

Running AE latent_dim=16

Running AE latent_dim=32

Running AE latent_dim=64

Running AE latent_dim=128
